## Artificial Neural Network with Backpropagation

**Backpropagation** is the most widely used algorithm for training artificial neural networks. It's an iterative process that adjusts the weights of the network by propagating the error backwards from the output layer to the input layer.

Here's a high-level overview of the steps:

1.  **Forward Pass**: Input data is fed into the network, and activations are computed layer by layer until the output layer is reached.
2.  **Compute Loss**: The output of the network is compared to the actual target values, and a loss (or error) is calculated.
3.  **Backward Pass (Backpropagation)**: The loss is then propagated backward through the network. The error contribution of each weight and bias is calculated using the chain rule of calculus.
4.  **Update Weights**: The weights and biases are adjusted in the direction that minimizes the loss, typically using an optimization algorithm like Gradient Descent.

We will implement a simple feedforward neural network from scratch using NumPy to demonstrate this process.

In [1]:
import numpy as np

# Define activation functions and their derivatives
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)


class NeuralNetwork:
    def __init__(self, layers, activation='sigmoid'):
        # `layers` is a list containing the number of neurons in each layer.
        # e.g., [2, 4, 1] for 2 input, 4 hidden, 1 output neuron
        self.layers = layers
        self.num_layers = len(layers)
        self.weights = []
        self.biases = []

        # Initialize weights and biases with random values
        for i in range(self.num_layers - 1):
            # Weights are initialized randomly from a standard normal distribution
            # scaled by sqrt(2 / n_in) for better convergence (He initialization for ReLU, Xavier for sigmoid/tanh).
            # Here, a common initialization is used.
            self.weights.append(np.random.randn(layers[i], layers[i+1]) * np.sqrt(2. / layers[i]))
            self.biases.append(np.zeros((1, layers[i+1])))

        # Set activation function and its derivative
        if activation == 'sigmoid':
            self.activation = sigmoid
            self.activation_derivative = sigmoid_derivative
        elif activation == 'tanh':
            self.activation = tanh
            self.activation_derivative = tanh_derivative
        elif activation == 'relu':
            self.activation = relu
            self.activation_derivative = relu_derivative
        else:
            raise ValueError("Unsupported activation function. Choose 'sigmoid', 'tanh', or 'relu'.")

    def forward_pass(self, X):
        # Stores activations for each layer, used in backpropagation
        self.activations = [X]
        A = X
        for i in range(self.num_layers - 1):
            # Z = A * W + B (matrix multiplication)
            Z = np.dot(A, self.weights[i]) + self.biases[i]
            A = self.activation(Z)
            self.activations.append(A)
        return A

    def backward_pass(self, X, y, output):
        # `output` is the result from the forward pass
        # `y` is the true target values

        # Calculate error at the output layer
        # For Mean Squared Error (MSE), the derivative is (output - y)
        error = y - output
        delta = error * self.activation_derivative(output)

        # Store gradients for weights and biases
        self.dw = [None] * (self.num_layers - 1)
        self.db = [None] * (self.num_layers - 1)

        # Backpropagate through the layers
        for i in range(self.num_layers - 2, -1, -1):
            # Gradient for weights
            self.dw[i] = np.dot(self.activations[i].T, delta)
            # Gradient for biases
            self.db[i] = np.sum(delta, axis=0, keepdims=True)

            # Calculate delta for the next layer backwards
            if i > 0:
                delta = np.dot(delta, self.weights[i].T) * self.activation_derivative(self.activations[i])

    def update_weights(self, learning_rate):
        for i in range(self.num_layers - 1):
            self.weights[i] += learning_rate * self.dw[i]
            self.biases[i] += learning_rate * self.db[i]

    def train(self, X, y, epochs, learning_rate):
        for epoch in range(epochs):
            output = self.forward_pass(X)
            self.backward_pass(X, y, output)
            self.update_weights(learning_rate)

            if epoch % (epochs // 10) == 0:
                loss = np.mean(np.square(y - output))
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        return self.forward_pass(X)

### Example Usage: XOR Problem

Let's test our Neural Network with the classic XOR problem. The XOR function is a binary operation that outputs true if its inputs are different, and false if they are the same. It's a simple non-linear problem that requires at least one hidden layer to solve.

In [2]:
# Input data for XOR (2 inputs)
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# Target output for XOR (1 output)
y = np.array([
    [0],
    [1],
    [1],
    [0]
])

# Define the neural network architecture:
# 2 input neurons, 4 hidden neurons, 1 output neuron
n_input = X.shape[1]
n_hidden = 4
n_output = y.shape[1]

# Create an instance of the Neural Network
# Using tanh activation for better performance on XOR
model = NeuralNetwork(layers=[n_input, n_hidden, n_output], activation='tanh')

# Train the network
epochs = 10000
learning_rate = 0.1
print("\nStarting training for XOR problem...")
model.train(X, y, epochs, learning_rate)

# Make predictions after training
predictions = model.predict(X)

# Convert predictions to binary output (0 or 1)
# Using a threshold of 0.5 for sigmoid, or 0 for tanh if output is centered around 0
if model.activation == sigmoid:
    binary_predictions = (predictions > 0.5).astype(int)
else: # For tanh, outputs are typically between -1 and 1
    binary_predictions = (predictions > 0).astype(int)

print("\n--- Training Complete ---")
print("\nInput:\n", X)
print("\nActual Output:\n", y)
print("\nPredicted Output:\n", binary_predictions)

# Evaluate accuracy
accuracy = np.mean(binary_predictions == y)
print(f"\nAccuracy: {accuracy * 100:.2f}%")


Starting training for XOR problem...
Epoch 0, Loss: 0.3137
Epoch 1000, Loss: 0.0001
Epoch 2000, Loss: 0.0021
Epoch 3000, Loss: 0.0015
Epoch 4000, Loss: 0.0012
Epoch 5000, Loss: 0.0010
Epoch 6000, Loss: 0.0009
Epoch 7000, Loss: 0.0008
Epoch 8000, Loss: 0.0007
Epoch 9000, Loss: 0.0006

--- Training Complete ---

Input:
 [[0 0]
 [0 1]
 [1 0]
 [1 1]]

Actual Output:
 [[0]
 [1]
 [1]
 [0]]

Predicted Output:
 [[0]
 [1]
 [1]
 [0]]

Accuracy: 100.00%
